# create_all_tables

Single DDL script — creates the full `spotify_etl` catalog schema:

| Layer | Tables |
|---|---|
| **raw** | `spotify_tokens`, `spotify_api_calls`, `spotify_api_errors` |
| **bronze** | `play_history`, `playlists`, `playlist_tracks`, `tracks`, `artists`, `audio_features` |
| **silver** | `fct_plays`, `dim_tracks`, `dim_artists`, `dim_playlists`, `dim_time` |
| **gold** | `user_listening_summary`, `top_tracks`, `top_artists`, `genre_trends`, `listening_patterns` |

Run once per environment. All statements use `CREATE TABLE IF NOT EXISTS` — idempotent.

In [ ]:
%run ../tools/config/settings

## Catalog & Schemas

In [ ]:
spark.sql(f"CREATE CATALOG  IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA   IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}")
spark.sql(f"CREATE SCHEMA   IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA   IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA   IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
print("Catalog and schemas ready.")

## Raw Layer — Monitoring tables

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}.spotify_tokens (
    token_name    STRING  NOT NULL,
    access_token  STRING,
    refresh_token STRING,
    expires_at    BIGINT,
    updated_at    TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed'   = 'true',
    'delta.columnMapping.mode'     = 'name'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}.spotify_api_calls (
    ingestion_ts          TIMESTAMP NOT NULL,
    ingestion_date        DATE      NOT NULL,
    run_id                STRING    NOT NULL,
    token_name            STRING,
    method                STRING    NOT NULL,
    endpoint              STRING,
    request_url           STRING    NOT NULL,
    request_params_json   STRING,
    request_body_json     STRING,
    http_status           INT,
    response_headers_json STRING,
    payload_json          STRING,
    error                 STRING,
    attempt               INT,
    request_hash          STRING    NOT NULL
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
    'delta.enableChangeDataFeed'           = 'true',
    'delta.autoOptimize.optimizeWrite'     = 'true',
    'delta.autoOptimize.autoCompact'       = 'true',
    'delta.columnMapping.mode'             = 'name'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}.spotify_api_errors (
    ingestion_ts          TIMESTAMP NOT NULL,
    ingestion_date        DATE      NOT NULL,
    run_id                STRING    NOT NULL,
    token_name            STRING,
    method                STRING    NOT NULL,
    endpoint              STRING,
    request_url           STRING    NOT NULL,
    request_params_json   STRING,
    request_body_json     STRING,
    http_status           INT,
    response_headers_json STRING,
    payload_json          STRING,
    error                 STRING,
    attempt               INT,
    request_hash          STRING    NOT NULL
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
    'delta.enableChangeDataFeed'           = 'true',
    'delta.autoOptimize.optimizeWrite'     = 'true',
    'delta.autoOptimize.autoCompact'       = 'true',
    'delta.columnMapping.mode'             = 'name'
)
""")

print("Raw tables ready.")

## Bronze Layer

In [ ]:
_BRONZE_OPTS = """
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'delta.columnMapping.mode'         = 'name'
)"""

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.bronze_play_history (
    played_at           TIMESTAMP   COMMENT 'UTC timestamp of the play event',
    track_id            STRING,
    track_name          STRING,
    track_duration_ms   BIGINT,
    artist_ids          ARRAY<STRING>,
    artist_names        ARRAY<STRING>,
    album_id            STRING,
    album_name          STRING,
    context_type        STRING      COMMENT 'playlist / artist / album / null',
    context_href        STRING,
    _raw                STRING      COMMENT 'Full serialized API item JSON',
    run_id              STRING,
    ingestion_date      DATE
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Bronze: raw listening history from /me/player/recently-played'
{_BRONZE_OPTS}
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.bronze_playlists (
    playlist_id         STRING,
    playlist_name       STRING,
    description         STRING,
    owner_id            STRING,
    owner_name          STRING,
    total_tracks        BIGINT,
    is_public           BOOLEAN,
    is_collaborative    BOOLEAN,
    _raw                STRING,
    run_id              STRING,
    ingestion_date      DATE
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Bronze: raw playlists from /me/playlists'
{_BRONZE_OPTS}
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.bronze_playlist_tracks (
    playlist_id         STRING      COMMENT 'Parent playlist ID (injected by pipeline)',
    track_id            STRING,
    track_name          STRING,
    added_at            TIMESTAMP,
    added_by            STRING,
    _raw                STRING,
    run_id              STRING,
    ingestion_date      DATE
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Bronze: raw playlist tracks from /playlists/{{id}}/tracks'
{_BRONZE_OPTS}
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.bronze_tracks (
    track_id            STRING,
    track_name          STRING,
    duration_ms         BIGINT,
    popularity          INT,
    explicit            BOOLEAN,
    artist_ids          ARRAY<STRING>,
    artist_names        ARRAY<STRING>,
    album_id            STRING,
    album_name          STRING,
    album_release_date  STRING      COMMENT 'Spotify partial-date string (YYYY / YYYY-MM / YYYY-MM-DD)',
    _raw                STRING,
    run_id              STRING,
    ingestion_date      DATE
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Bronze: raw track objects from /tracks'
{_BRONZE_OPTS}
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.bronze_artists (
    artist_id           STRING,
    artist_name         STRING,
    genres              ARRAY<STRING>,
    popularity          INT,
    followers_total     BIGINT,
    _raw                STRING,
    run_id              STRING,
    ingestion_date      DATE
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Bronze: raw artist objects from /artists'
{_BRONZE_OPTS}
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.bronze_audio_features (
    track_id            STRING,
    danceability        DOUBLE      COMMENT '0.0–1.0',
    energy              DOUBLE      COMMENT '0.0–1.0',
    key                 INT         COMMENT 'Pitch class 0–11',
    loudness            DOUBLE      COMMENT 'dB, typically -60 to 0',
    mode                INT         COMMENT '1=major, 0=minor',
    speechiness         DOUBLE,
    acousticness        DOUBLE,
    instrumentalness    DOUBLE,
    liveness            DOUBLE,
    valence             DOUBLE      COMMENT 'Musical positivity 0.0–1.0',
    tempo               DOUBLE      COMMENT 'BPM',
    duration_ms         BIGINT,
    time_signature      INT,
    _raw                STRING,
    run_id              STRING,
    ingestion_date      DATE
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Bronze: raw audio features from /audio-features'
{_BRONZE_OPTS}
""")

print("Bronze tables ready.")

## Silver Layer

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.fct_plays (
    play_id             STRING      NOT NULL COMMENT 'SHA256(played_at||track_id)',
    played_at           TIMESTAMP   NOT NULL,
    track_id            STRING      NOT NULL,
    ingestion_date      DATE,
    run_id              STRING
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Silver fact: deduplicated play events'
TBLPROPERTIES (
    'delta.enableChangeDataFeed'       = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.columnMapping.mode'         = 'name'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.dim_tracks (
    track_id            STRING      NOT NULL,
    track_name          STRING,
    duration_ms         BIGINT,
    duration_min        DOUBLE,
    popularity          INT,
    explicit            BOOLEAN,
    album_id            STRING,
    album_name          STRING,
    album_release_date  DATE        COMMENT 'Parsed from Spotify partial-date string',
    _valid_from         DATE
)
USING DELTA
COMMENT 'Silver dimension: normalized track attributes (SCD Type 1)'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.columnMapping.mode'   = 'name'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.dim_artists (
    artist_id           STRING      NOT NULL,
    artist_name         STRING,
    genres              ARRAY<STRING>,
    primary_genre       STRING      COMMENT 'genres[0], null if array is empty',
    popularity          INT,
    followers_total     BIGINT,
    _valid_from         DATE
)
USING DELTA
COMMENT 'Silver dimension: artist attributes (SCD Type 1)'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.columnMapping.mode'   = 'name'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.dim_playlists (
    playlist_id         STRING      NOT NULL,
    playlist_name       STRING,
    description         STRING,
    owner_id            STRING,
    owner_name          STRING,
    total_tracks        BIGINT,
    is_public           BOOLEAN,
    is_collaborative    BOOLEAN,
    _valid_from         DATE
)
USING DELTA
COMMENT 'Silver dimension: playlist attributes (SCD Type 1)'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.columnMapping.mode'   = 'name'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.dim_time (
    played_at           TIMESTAMP   NOT NULL,
    play_date           DATE,
    play_hour           INT         COMMENT '0–23',
    day_of_week         INT         COMMENT '1=Sunday … 7=Saturday (Spark default)',
    day_name            STRING,
    week_of_year        INT,
    month               INT,
    month_name          STRING,
    year                INT,
    is_weekend          BOOLEAN
)
USING DELTA
COMMENT 'Silver dimension: time attributes derived from play timestamps'
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name'
)
""")

print("Silver tables ready.")

## Gold Layer

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.gold_user_listening_summary (
    snapshot_date       DATE        NOT NULL,
    total_plays         BIGINT,
    unique_tracks       BIGINT,
    unique_artists      BIGINT,
    total_minutes       DOUBLE
)
USING DELTA
COMMENT 'Gold: cumulative listening summary per snapshot date'
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.gold_top_tracks (
    snapshot_date       DATE        NOT NULL,
    period              STRING      NOT NULL COMMENT '7d / 30d / all',
    rank                INT         NOT NULL,
    track_id            STRING,
    track_name          STRING,
    play_count          BIGINT
)
USING DELTA
PARTITIONED BY (snapshot_date)
COMMENT 'Gold: top tracks ranked by play count per period'
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.gold_top_artists (
    snapshot_date       DATE        NOT NULL,
    period              STRING      NOT NULL COMMENT '7d / 30d / all',
    rank                INT         NOT NULL,
    artist_id           STRING,
    artist_name         STRING,
    play_count          BIGINT
)
USING DELTA
PARTITIONED BY (snapshot_date)
COMMENT 'Gold: top artists ranked by play count per period'
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.gold_genre_trends (
    snapshot_date       DATE        NOT NULL,
    genre               STRING      NOT NULL,
    play_count          BIGINT,
    unique_tracks       BIGINT
)
USING DELTA
PARTITIONED BY (snapshot_date)
COMMENT 'Gold: genre play counts per snapshot date'
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.gold_listening_patterns (
    snapshot_date       DATE        NOT NULL,
    hour_of_day         INT         NOT NULL COMMENT '0–23',
    day_of_week         INT         NOT NULL COMMENT '1=Sunday … 7=Saturday',
    day_name            STRING,
    play_count          BIGINT
)
USING DELTA
PARTITIONED BY (snapshot_date)
COMMENT 'Gold: play count distribution by hour and day-of-week'
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
""")

print("Gold tables ready.")

## Summary

In [ ]:
tables = spark.sql(f"""
    SELECT table_schema AS layer, table_name, table_type
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema IN ('{RAW_SCHEMA}', '{BRONZE_SCHEMA}', '{SILVER_SCHEMA}', '{GOLD_SCHEMA}')
    ORDER BY table_schema, table_name
""")
display(tables)